# Hybrid Deepfake Detector — One-Day Prototype Run (Colab T4)

This notebook runs the existing `yashsidana/hybrid-deepfake-detector` pipeline end to end on a **T4 GPU** (Runtime → Change runtime type → T4 GPU). It does **not** redesign anything — every step below just calls scripts already in the repo (plus a few small additions made for this run: dataset subsetting, a cross-dataset split script, and an ablation harness).

**What this notebook does, in order:**
1. Mount Google Drive (so hours of downloads/preprocessing survive a disconnect — Colab's local disk is wiped on disconnect).
2. Clone the repo, install dependencies.
3. Kaggle auth, download Celeb-DF v2 + the DFD (Deep Fake Detection) dataset.
4. Generate metadata, subset to ~1000 real / ~1000 fake per dataset, create identity-aware splits.
5. Precompute faces / temporal sequences / forensic features (cached, resumable).
6. Train the semantic branch (EfficientNet-B0, fine-tuned) and temporal branch (ResNet-18+LSTM, fine-tuned).
7. Extract embeddings, train the fusion SVM, run the final test evaluation.
8. Run the ablation study (Model A/B/C).
9. Run a strict cross-dataset generalization experiment (train on DFDC, test on Celeb-DF).
10. Zip up all results/checkpoints for download.

**Before you start:**
- Create a Kaggle API token: kaggle.com → Settings → Create New API Token → downloads `kaggle.json`.
- Note: this notebook uses DFD (Google/Jigsaw’s Deep Fake Detection dataset), not DFDC. DFDC’s own Kaggle competition has late submissions permanently disabled by the host, so its accept-rules step cannot be completed by any account anymore -- confirmed by testing directly. DFD is a plain public Kaggle Dataset (not a gated Competition), so it needs only a normal Kaggle login, no accept-rules step.
- Everything is designed to be resumable — if Colab disconnects, re-run from the top; already-downloaded/cached files are skipped, not re-fetched.

## 1. Mount Google Drive (persist data across disconnects)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/hybrid-deepfake-detector-run"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Persistent storage at:", DRIVE_ROOT)

Mounted at /content/drive
Persistent storage at: /content/drive/MyDrive/hybrid-deepfake-detector-run


## 2. Check GPU

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Expect: Tesla T4, ~15360 MiB. If this errors, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 3. Clone the repo and install dependencies

In [3]:
%cd /content
!git clone https://github.com/yashsidana/hybrid-deepfake-detector.git
%cd /content/hybrid-deepfake-detector
!pip install -q -r requirements.txt

/content
Cloning into 'hybrid-deepfake-detector'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 225 (delta 92), reused 147 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 126.89 KiB | 9.76 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/hybrid-deepfake-detector
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.4 MB/s eta 0:00:00


## 4. Point the repo's data/model directories at Drive

Symlinking these into Drive means downloads, precomputed caches, and checkpoints survive a Colab disconnect — re-running a cell after a reconnect resumes instead of starting over.

In [4]:
import os

for rel_dir in ["data/raw", "data/processed", "data/metadata", "data/splits",
                "models", "saved_models"]:
    drive_path = os.path.join(DRIVE_ROOT, rel_dir)
    os.makedirs(drive_path, exist_ok=True)
    local_path = os.path.join("/content/hybrid-deepfake-detector", rel_dir)
    if os.path.islink(local_path):
        continue
    if os.path.exists(local_path):
        import shutil
        shutil.rmtree(local_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    os.symlink(drive_path, local_path)
    print(f"{local_path} -> {drive_path}")

/content/hybrid-deepfake-detector/data/raw -> /content/drive/MyDrive/hybrid-deepfake-detector-run/data/raw
/content/hybrid-deepfake-detector/data/processed -> /content/drive/MyDrive/hybrid-deepfake-detector-run/data/processed
/content/hybrid-deepfake-detector/data/metadata -> /content/drive/MyDrive/hybrid-deepfake-detector-run/data/metadata
/content/hybrid-deepfake-detector/data/splits -> /content/drive/MyDrive/hybrid-deepfake-detector-run/data/splits
/content/hybrid-deepfake-detector/models -> /content/drive/MyDrive/hybrid-deepfake-detector-run/models
/content/hybrid-deepfake-detector/saved_models -> /content/drive/MyDrive/hybrid-deepfake-detector-run/saved_models


## 5. Kaggle authentication

Run this cell, then upload the `kaggle.json` you downloaded from kaggle.com/settings when prompted.

In [5]:
from google.colab import files
import os

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select your kaggle.json
for fname in uploaded:
    if fname != "kaggle.json":
        os.rename(fname, "kaggle.json")
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle credentials installed.")

Saving kaggle (1).json to kaggle (1).json
Kaggle credentials installed.


## 6. Download datasets

Both Celeb-DF v2 and DFD are plain public Kaggle Datasets (not gated Competitions), so both are fully automated -- no manual accept-rules step, unlike DFDC (whose Kaggle competition has late submissions permanently disabled by the host) or FaceForensics++/FakeAVCeleb (manual Data Use Agreement review, days to a week).

In [6]:
from src.data.adapters.celebdf import CelebDFAdapter
from src.data.adapters.dfd import DFDAdapter

print("Downloading Celeb-DF v2 (fully automated)...")
CelebDFAdapter().download()

print("Downloading DFD (Google/Jigsaw Deep Fake Detection dataset, fully automated)...")
DFDAdapter().download()
# ~24GB total (original + manipulated sequences). No manual approval step.

Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'deep-fake-detection-dfd-entire-original-dataset' dataset.


In [7]:
import json
with open('/root/.kaggle/kaggle.json') as f:
    d = json.load(f)
    print(list(d.keys()))

['username', 'key']


## 7. Generate metadata, subset to target counts, create splits

In [8]:
import json
raw = open('/root/.kaggle/kaggle.json', 'rb').read().decode('utf-8-sig')
outer = json.loads(raw)
inner_key = list(outer.keys())[0]
fixed = json.loads(inner_key.lstrip('﻿'))
assert 'username' in fixed and 'key' in fixed
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(fixed, f)
    import os
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Fixed. Keys:', list(fixed.keys()))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
!python -m src.data.metadata

[celebdf] Scanning videos...
[ WARN:0@158.033] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 64409.759265 ms
[celebdf] 6529 videos found.
[dfd] Scanning videos...
[ WARN:0@635.079] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 204549.077260 ms
[dfd] 3431 videos found.

Metadata written to data/metadata/metadata.csv
Total videos: 9960
Per-dataset counts:
dataset
celebdf    6529
dfd        3431
Per-label counts:
label
1    8707
0    1253


In [10]:
exec("import os\nrp = '/content/hybrid-deepfake-detector/data/raw'\nprint('is symlink:', os.path.islink(rp))\nprint('celebdf real:', len(os.listdir(rp+'/celebdf/real')) if os.path.isdir(rp+'/celebdf/real') else 'MISSING')\nprint('celebdf fake:', len(os.listdir(rp+'/celebdf/fake')) if os.path.isdir(rp+'/celebdf/fake') else 'MISSING')\nimport subprocess\nprint(subprocess.run(['df','-h','/content'], capture_output=True, text=True).stdout)")

is symlink: True
celebdf real: 890
celebdf fake: 5639
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   60G   54G  53% /



In [12]:
exec("import os, shutil, subprocess\nrepo_root = '/content/hybrid-deepfake-detector'\ndrive_raw = os.readlink(os.path.join(repo_root, 'data/raw'))\nprint('Drive raw path:', drive_raw)\nfor sub in ['celebdf','dfd']:\n    p = os.path.join(drive_raw, sub)\n    if os.path.isdir(p):\n        n = sum(len(fs) for _,_,fs in os.walk(p))\n        print(sub, 'file count on Drive:', n)\n    else:\n        print(sub, 'not present on Drive')")

Drive raw path: /content/drive/MyDrive/hybrid-deepfake-detector-run/data/raw
celebdf file count on Drive: 6529
dfd file count on Drive: 3431


In [16]:
exec("import os, shutil\nrepo_root = '/content/hybrid-deepfake-detector'\ndrive_root = os.readlink(os.path.join(repo_root, 'data/raw'))\nlocal_raw = os.path.join(repo_root, 'data/raw')\nos.unlink(local_raw)\nshutil.copytree(drive_root, local_raw)\nprint('Copied data/raw from Drive to local.')\nlocal_proc = os.path.join(repo_root, 'data/processed')\nif os.path.islink(local_proc):\n    os.unlink(local_proc)\nelif os.path.exists(local_proc):\n    shutil.rmtree(local_proc)\nos.makedirs(local_proc, exist_ok=True)\nprint('data/processed is now LOCAL (non-Drive).')\nfor sub in ['celebdf','dfd']:\n    p = os.path.join(local_raw, sub)\n    n = sum(len(fs) for _,_,fs in os.walk(p)) if os.path.isdir(p) else 0\n    print(sub, 'local file count:', n)")

OSError: [Errno 22] Invalid argument: '/content/hybrid-deepfake-detector/data/raw'

In [15]:
print('Checking Google Drive mount status:')
!ls /content/drive

Checking Google Drive mount status:
MyDrive


Check the per-dataset/per-label counts printed above. DFD has ~3,068 manipulated and ~363 original-actor videos (spread across multiple action clips each), so real/fake balance may differ from an even split -- that's expected, subset_metadata.py below reports the actual achievable counts honestly rather than forcing 1000/1000.

In [18]:
!python -m src.preprocessing.subset_metadata --target celebdf:1000:1000 --target dfd:1000:1000
# Read the "shortfall" section of the output -- if a target wasn't fully met,
# that's expected/honest, not an error. It reports actual achieved counts.

Backed up full metadata (9960 rows) to data/metadata/metadata_full.csv

Subsetting report:
dataset label  requested  available  kept  shortfall
celebdf  real       1000        890   890        110
celebdf  fake       1000       5639  1000          0
    dfd  real       1000        363   363        637
    dfd  fake       1000       3068  1000          0

NOTE: the following groups had fewer videos available than requested -- kept everything available rather than forcing the target:
dataset label  requested  available  kept  shortfall
celebdf  real       1000        890   890        110
    dfd  real       1000        363   363        637

data/metadata/metadata.csv now has 3253 rows (was 9960).
Per-dataset/label counts after subsetting:
dataset  label
celebdf  0         890
         1        1000
dfd      0         363
         1        1000


In [19]:
!python -m src.preprocessing.create_splits
# Identity-aware, leakage-safe, ~80/10/10. This is the MIXED-domain split
# (both datasets pooled) used for the main prototype result and the
# ablation study. The strict cross-dataset split happens later, separately.

Splits created successfully (identity-aware, multi-dataset).
Train: 3133  (groups: 306)
Val:   1  (groups: 1)
Test:  119  (groups: 1)

Per-dataset breakdown:
  train: {'celebdf': 1889, 'dfd': 1244}
  val: {'celebdf': 1}
  test: {'dfd': 119}


In [21]:
exec("import importlib, pandas as pd\nimport src.preprocessing.create_splits as cs\nimportlib.reload(cs)\ndf = pd.read_csv(cs.METADATA_PATH)\ndf['_group'] = cs._assign_groups(df)\nsizes = df.groupby('_group').size().sort_values(ascending=False)\nprint('Num groups:', len(sizes), 'Total rows:', len(df))\nprint('Top 10 group sizes:', sizes.head(10).tolist())\nbest = None\nfor seed in range(0, 25):\n    tr, va, te = cs._stratified_group_split(df, group_col='_group', label_col='label', test_fraction=cs.TEST_FRACTION, val_fraction=cs.VAL_FRACTION, seed=seed)\n    val_ok = len(va) >= 80 and va['label'].nunique() == 2\n    test_ok = len(te) >= 80 and te['label'].nunique() == 2\n    tag = 'USABLE' if (val_ok and test_ok) else ''\n    print(f'seed={seed} train={len(tr)} val={len(va)} test={len(te)} {tag}')\n    if val_ok and test_ok:\n        score = min(len(va), len(te))\n        if best is None or score > best[0]:\n            best = (score, seed)\nprint('BEST:', best)")

Num groups: 308 Total rows: 3253
Top 10 group sizes: [1095, 1094, 217, 194, 150, 119, 74, 10, 1, 1]
seed=0 train=1917 val=1335 test=1 
seed=1 train=1917 val=1335 test=1 
seed=2 train=3010 val=242 test=1 
seed=3 train=1917 val=1335 test=1 
seed=4 train=1917 val=1335 test=1 
seed=5 train=1917 val=1335 test=1 
seed=6 train=1917 val=1335 test=1 
seed=7 train=1917 val=1335 test=1 
seed=8 train=1917 val=1335 test=1 
seed=9 train=1917 val=1335 test=1 
seed=10 train=1917 val=1335 test=1 
seed=11 train=1917 val=1335 test=1 
seed=12 train=1917 val=1335 test=1 
seed=13 train=1917 val=1335 test=1 
seed=14 train=1917 val=1335 test=1 
seed=15 train=1917 val=1335 test=1 
seed=16 train=1917 val=1335 test=1 
seed=17 train=1917 val=1335 test=1 
seed=18 train=1917 val=1335 test=1 
seed=19 train=1917 val=1335 test=1 
seed=20 train=1917 val=1335 test=1 
seed=21 train=1917 val=1335 test=1 
seed=22 train=3033 val=219 test=1 
seed=23 train=1917 val=1335 test=1 
seed=24 train=1917 val=1335 test=1 
BEST: None


In [22]:
exec("from sklearn.model_selection import StratifiedGroupKFold\nbest_test = None\nfor n_splits in [3,4,5,6]:\n    for seed in range(6):\n        skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        for fold_i, (trv_idx, te_idx) in enumerate(skf.split(df, df['label'], groups=df['_group'])):\n            te = df.iloc[te_idx]\n            frac = len(te)/len(df)\n            if 0.15 <= frac <= 0.45 and te['label'].nunique() == 2:\n                print(f'n_splits={n_splits} seed={seed} fold={fold_i} test_frac={frac:.2f} test_n={len(te)}')\n                if best_test is None:\n                    best_test = (n_splits, seed, fold_i, len(te))\nprint('best_test:', best_test)")

n_splits=3 seed=0 fold=1 test_frac=0.40 test_n=1305
n_splits=3 seed=1 fold=1 test_frac=0.40 test_n=1305
n_splits=3 seed=2 fold=1 test_frac=0.40 test_n=1305
n_splits=3 seed=3 fold=1 test_frac=0.40 test_n=1305
n_splits=3 seed=4 fold=1 test_frac=0.40 test_n=1305
n_splits=3 seed=5 fold=1 test_frac=0.40 test_n=1305
n_splits=4 seed=0 fold=0 test_frac=0.40 test_n=1285
n_splits=4 seed=1 fold=0 test_frac=0.40 test_n=1285
n_splits=4 seed=2 fold=0 test_frac=0.40 test_n=1285
n_splits=4 seed=3 fold=0 test_frac=0.40 test_n=1285
n_splits=4 seed=4 fold=0 test_frac=0.40 test_n=1285
n_splits=4 seed=5 fold=0 test_frac=0.40 test_n=1285
n_splits=5 seed=0 fold=3 test_frac=0.41 test_n=1319
n_splits=5 seed=1 fold=3 test_frac=0.41 test_n=1319
n_splits=5 seed=2 fold=3 test_frac=0.41 test_n=1319
n_splits=5 seed=3 fold=3 test_frac=0.41 test_n=1319
n_splits=5 seed=4 fold=3 test_frac=0.41 test_n=1319
n_splits=5 seed=5 fold=3 test_frac=0.41 test_n=1319
n_splits=6 seed=0 fold=3 test_frac=0.39 test_n=1274
n_splits=6 s

In [23]:
exec("skf_test = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=0)\nfolds = list(skf_test.split(df, df['label'], groups=df['_group']))\ntrainval_idx, test_idx = folds[5]\ntrainval_df = df.iloc[trainval_idx]\ntest_df = df.iloc[test_idx]\nprint('test:', len(test_df), test_df['label'].value_counts().to_dict())\nbest_val = None\nfor n_splits in [4,5,6,7,8,10]:\n    for seed in range(6):\n        skf_val = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        for fold_i, (tr_idx, va_idx) in enumerate(skf_val.split(trainval_df, trainval_df['label'], groups=trainval_df['_group'])):\n            va = trainval_df.iloc[va_idx]\n            frac = len(va)/len(trainval_df)\n            if 0.08 <= frac <= 0.25 and va['label'].nunique() == 2:\n                if best_val is None or abs(frac-0.1) < abs(best_val[4]-0.1):\n                    best_val = (n_splits, seed, fold_i, len(va), frac)\nprint('best_val:', best_val)")

test: 1181 {1: 851, 0: 330}
best_val: (8, 0, 2, 209, 0.10086872586872588)


In [24]:
exec("skf_val = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=0)\nval_folds = list(skf_val.split(trainval_df, trainval_df['label'], groups=trainval_df['_group']))\ntrain_idx, val_idx = val_folds[2]\ntrain_df = trainval_df.iloc[train_idx]\nval_df = trainval_df.iloc[val_idx]\ntrain_groups = set(train_df['_group']); val_groups = set(val_df['_group']); test_groups = set(test_df['_group'])\noverlap = (train_groups & val_groups) | (train_groups & test_groups) | (val_groups & test_groups)\nassert not overlap, f'LEAKAGE DETECTED: {len(overlap)} overlapping groups'\nprint('No group overlap between train/val/test -- leakage-safe, confirmed.')\noutput_cols = ['video_path', 'dataset', 'identity', 'label']\nos.makedirs(cs.SPLITS_DIR, exist_ok=True)\ntrain_df[output_cols].to_csv(os.path.join(cs.SPLITS_DIR, 'train.csv'), index=False)\nval_df[output_cols].to_csv(os.path.join(cs.SPLITS_DIR, 'val.csv'), index=False)\ntest_df[output_cols].to_csv(os.path.join(cs.SPLITS_DIR, 'test.csv'), index=False)\nprint('Splits written to', cs.SPLITS_DIR)\nprint(f'Train: {len(train_df)}  (groups: {len(train_groups)})')\nprint(f'Val:   {len(val_df)}  (groups: {len(val_groups)})')\nprint(f'Test:  {len(test_df)}  (groups: {len(test_groups)})')\nprint(chr(10)+'Per-dataset breakdown:')\nfor name, d in (('train', train_df), ('val', val_df), ('test', test_df)):\n    print(f'  {name}:', d.groupby(['dataset','label']).size().to_dict())")

No group overlap between train/val/test -- leakage-safe, confirmed.
Splits written to data/splits
Train: 1863  (groups: 213)
Val:   209  (groups: 16)
Test:  1181  (groups: 79)

Per-dataset breakdown:
  train: {('celebdf', 0): 697, ('celebdf', 1): 897, ('dfd', 0): 120, ('dfd', 1): 149}
  val: {('celebdf', 0): 106, ('celebdf', 1): 103}
  test: {('celebdf', 0): 87, ('dfd', 0): 243, ('dfd', 1): 851}


## 8. Preprocessing: face crops, temporal sequences, forensic features

Each of these is resumable (skips files already cached) -- safe to re-run after a disconnect. This is also usually the slowest part of the whole pipeline (video decode + MTCNN face detection), not the GPU training steps.

In [25]:
!python -m src.preprocessing.precompute_faces

Precomputing semantic faces:  67% 2188/3253 [02:48<00:14, 73.73it/s][mov,mp4,m4a,3gp,3g2,mj2 @ 0x221e0a00] moov atom not found
Precomputing semantic faces: 100% 3253/3253 [02:49<00:00, 19.16it/s]  
Skipped 1328 video(s) -- see above / run validate.py for details.


In [26]:
!python -m src.preprocessing.precompute_temporal

Precomputing temporal sequences:  67% 2189/3253 [46:54<25:13,  1.42s/it][mov,mp4,m4a,3gp,3g2,mj2 @ 0xeee7b40] moov atom not found
Precomputing temporal sequences: 100% 3253/3253 [47:55<00:00,  1.13it/s]
Skipped 1328 video(s) -- see above / run validate.py for details.

Temporal face sequences cached to: data/processed/temporal


In [ ]:
!python -m src.preprocessing.precompute_forensic

## 9. Train the semantic branch (EfficientNet-B0, fine-tuned)

Batch 32, up to 10 epochs, validation Macro-F1 model selection. The backbone is ImageNet-pretrained but fine-tuned (not frozen), with a 10x-lower learning rate than the head so a few epochs on this dataset size doesn't wash out the pretrained features.

In [ ]:
!python -m src.modeling.train_semantic

## 10. Train the temporal branch (ResNet-18 + LSTM, fine-tuned)

Batch 16, up to 10 epochs, early stopping (patience 5), ReduceLROnPlateau, validation Macro-F1 model selection. If you hit CUDA OOM here, it's almost always this branch (16 frames x batch_size images through ResNet-18 with gradients now flowing through the backbone) -- lowering `BATCH_SIZE` at the top of `src/modeling/train_temporal.py` to 8 is the first thing to try.

In [ ]:
!python -m src.modeling.train_temporal

## 11. Extract embeddings (semantic + temporal + forensic) for the mixed-domain split

In [ ]:
!python -m src.modeling.extract_embeddings
# Writes data/processed/fusion/{train,val,test}.npz
# 256 (semantic) + 256 (temporal) + 63 (forensic) = 575-D per video.

## 12. Train the fusion classifier (distribution matching + RBF SVM)

Distribution matcher fit on TRAIN "real" samples only. SVM hyperparameters (C in [0.1, 1, 10, 100]) chosen by GridSearchCV scored on VALIDATION Macro-F1 -- the test set is not touched here.

In [ ]:
!python -m src.modeling.train_fusion

## 13. Final test-set evaluation (one-time, this is the headline result)

In [ ]:
!python -m src.modeling.test_fusion
# Reports: Accuracy, Precision, Recall, F1, Macro-F1, Balanced Accuracy,
# ROC-AUC, confusion matrix, classification report, real/fake test counts,
# false positives/negatives. Saved to saved_models/test_fusion_evaluation_report.json

## 14. Ablation study (Model A / B / C)

Reuses the same embeddings extracted in Section 11 -- no re-extraction needed. Each model gets its own freshly-fit distribution matcher/scaler/SVM on its own branch subset (excluded branches are genuinely absent from the fused vector, not zeroed-in).

In [ ]:
!python -m src.modeling.run_ablation

## 15. Cross-dataset generalization: TRAIN on DFD, TEST on Celeb-DF v2

Strict cross-dataset evaluation -- Celeb-DF v2 samples are used ONLY for the final test call below, never for training, validation, scaler fitting, distribution estimation, or SVM model selection. This answers a different question than Sections 12-13 (which mix both datasets): does the model generalize to a completely unseen dataset's generation methods, not just unseen identities within a domain it has already seen.

In [ ]:
!python -m src.preprocessing.create_cross_dataset_splits \
    --train-dataset dfd --test-dataset celebdf \
    --out-dir data/splits_cross_dfd_to_celebdf

In [ ]:
!python -m src.modeling.extract_embeddings \
    --splits-root data/splits_cross_dfd_to_celebdf \
    --output-root data/processed/fusion_cross_dfd_to_celebdf

In [ ]:
!python -m src.modeling.train_fusion \
    --embeddings-root data/processed/fusion_cross_dfd_to_celebdf \
    --model-path models/fusion_classifier/fusion_model_cross_dfd_to_celebdf.pkl \
    --report-path saved_models/train_fusion_report_cross_dfd_to_celebdf.json

In [ ]:
!python -m src.modeling.test_fusion \
    --embeddings-root data/processed/fusion_cross_dfd_to_celebdf \
    --model-path models/fusion_classifier/fusion_model_cross_dfd_to_celebdf.pkl \
    --report-path saved_models/test_fusion_evaluation_report_cross_dfd_to_celebdf.json

## 16. (Optional, if time/compute budget remains) Reverse direction: TRAIN on Celeb-DF, TEST on DFD

Only run this if Sections 1-15 finished with GPU time to spare -- it's not required, and the spec explicitly says not to sacrifice the core prototype for extra experiments.

In [ ]:
!python -m src.preprocessing.create_cross_dataset_splits \
    --train-dataset celebdf --test-dataset dfd \
    --out-dir data/splits_cross_celebdf_to_dfd

!python -m src.modeling.extract_embeddings \
    --splits-root data/splits_cross_celebdf_to_dfd \
    --output-root data/processed/fusion_cross_celebdf_to_dfd

!python -m src.modeling.train_fusion \
    --embeddings-root data/processed/fusion_cross_celebdf_to_dfd \
    --model-path models/fusion_classifier/fusion_model_cross_celebdf_to_dfd.pkl \
    --report-path saved_models/train_fusion_report_cross_celebdf_to_dfd.json

!python -m src.modeling.test_fusion \
    --embeddings-root data/processed/fusion_cross_celebdf_to_dfd \
    --model-path models/fusion_classifier/fusion_model_cross_celebdf_to_dfd.pkl \
    --report-path saved_models/test_fusion_evaluation_report_cross_celebdf_to_dfd.json

## 17. Collect and zip results for download

Everything under `saved_models/` (JSON reports, .pth checkpoints) and `models/` (fusion_model*.pkl) is already safe in Drive via the Section 4 symlinks. This cell also makes one zip for convenience.

In [ ]:
import shutil
shutil.make_archive("/content/drive/MyDrive/hybrid-deepfake-detector-run/results_bundle",
                     "zip", "/content/hybrid-deepfake-detector/saved_models")
print("Zipped saved_models/ to Drive: hybrid-deepfake-detector-run/results_bundle.zip")

print("\n--- Summary of report files ---")
!find saved_models -name "*.json" | sort
!find models -name "*.pkl" | sort

## 18. (When ready) Push code/results back to GitHub

Only run this if you want to commit anything changed here (e.g. if you tuned a hyperparameter directly in Colab). The result *reports* are just JSON/pkl files in Drive from Section 17 -- you don't need to push those to share them, you can just download `results_bundle.zip` from Drive.

In [ ]:
# %cd /content/hybrid-deepfake-detector
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add -A
# !git commit -m "Prototype run results"
# !git push https://<username>:<token>@github.com/yashsidana/hybrid-deepfake-detector.git main